In [1]:
# Pip install tings and import tings
!pip3 install beautifulsoup4
!pip3 install requests
!pip3 install serpapi
from bs4 import BeautifulSoup
import requests
import json
from datetime import datetime
import dateutil.parser as parser

This block defines a function that gets all of the specified newspapers' stories that include the given keyword in their headline

In [ ]:
#This dictionary is used by the following function - if paper not recognised [] is returned
# This uses SerpAPI
from bs4 import BeautifulSoup
import requests
import serpapi
import json

papers = {
    'guardian': "www.theguardian.com",
    'dailymail' : "www.dailymail.co.uk"
}

# return a list of articles of form
"""
{headline : text, body: text, date: date, author: author}
"""
def get_all_articles(paper, keyword):
  # Check that paper exists
  if paper.lower() not in papers:
    return []
  searches = []
  end = False

  start = 0

  while(not end):


    params = {
      "q": 'allintitle:"' + keyword + '" site:' + papers[paper] + '',
      "location": "London, United Kingdom",
      "hl": "en",
      "gl": "gb",
      "google_domain": "google.com",
      "api_key": "2603149264d4eac3fc8daef51a3f3f850a97e2dbd117da04880691c595b80274",
      "start": start,
      "output": 'json'
    }

    search = serpapi.search(params).as_dict()
    # Check if any more results have been fetched
    if(search['search_information']['organic_results_state'] == 'Fully empty'):
      end = True
    else:
      searches.append(search)
      start += 10
      print("Done page " + str(start/10))
  return searches

# fake paper name not to drain SERPAPI requests in case I accidentally hate this.
paper_name = 'dailymailAAA'
searchResults = (get_all_articles(paper_name, 'uber'))

json_object = json.dumps(searchResults, indent=4)
with open("raw_" + paper_name + ".json", "w") as outfile:
    outfile.write(json_object)


Done page 1.0
Done page 2.0
Done page 3.0
Done page 4.0
Done page 5.0
Done page 6.0
Done page 7.0
Done page 8.0
Done page 9.0
Done page 10.0
Done page 11.0
Done page 12.0
Done page 13.0
Done page 14.0
Done page 15.0
Done page 16.0
Done page 17.0
Done page 18.0
Done page 19.0
Done page 20.0
Done page 21.0
Done page 22.0
Done page 23.0
Done page 24.0
Done page 25.0
Done page 26.0
Done page 27.0
Done page 28.0
Done page 29.0
Done page 30.0


In [ ]:

# This block of code will convert the extant JSON files into a the final one for analysis (as shown in Template.JSON)

# DAILY MAIL ARTICLE -> FORMAT
# Sanity checks
# Only one headline
# Only one (group) of authors
# Only one date

# First time accessing an article

def read_dailymail_article(url):
  err_obj = {
      'url': url
  }
  # all daily mail articles contain article in url so ignore non-articles
  if 'article' not in url:
    err_obj['failed'] = 'non-article url'
    return err_obj

  # Requests need the url of the page to access
  site = requests.get(url)

  # Beautiful Soup takes the text from the page accessed by requests and makes it easy to use
  # 'html.parser' is the standard way to process HTML in python
  soup = BeautifulSoup(site.text, 'html.parser')

  # Get headline - CHECK THAT THERE IS ONLY ONE HEADLINE


  headline = soup.find_all('h1')
  if len(headline) != 1:
    err_obj['failed'] = 'headline'
    return err_obj

  headline = headline[0].get_text().lower()

  # sometimes headline starts with exclusive without a space, so add a space if so
  if headline.startswith('exclusive'):
    headline = 'exclusive ' + headline.split('exclusive', maxsplit=2)[1].strip()

 # there can be multiple authors
  authors = soup.find_all(class_="author")
  # sometimes authors stored in this tag author-section byline-plain so try this too
  by_for = False
  if len(authors) == 0:
    by_for = True
    authors = soup.find_all(class_="author-section byline-plain")

  # sometimes this tag is used to store multiple authors separated by AND author-section mol-para-with-font byline-plain
  and_separator = False
  if len(authors) == 0:
    and_separator = True
    by_for = False
    authors = soup.find_all(class_="author-section mol-para-with-font byline-plain")

  # check there's at least one
  if len(authors) == 0:
    err_obj['failed'] = 'author'
    return err_obj

  authors = [a.get_text().lower().strip() for a in authors]
  # and_separator means multiple authors separated by and
  if(by_for):
    authors_tmp = authors
    authors = []
    for author in authors_tmp:
      [authors.append(a.strip()) for a in author.split("by ", maxsplit=2)[1].split(' for ', maxsplit=2)[0].split(' and ')]

  if(and_separator):
    authors_tmp = authors
    authors = []
    for author in authors_tmp:
      [authors.append(a.strip()) for a in author.split("by ", maxsplit=2)[1].split(' and ')]

  # Dailymail has published and updated date let's record both
  # Get published time
  published = soup.find(class_ = 'article-timestamp article-timestamp-published')
  if published is None:
    err_obj['failed'] = 'published'
    return err_obj
  published = published.find('time').attrs['datetime']
  published = parser.parse(published)
  # Convert to datetime
  # Get updated time
  updated = soup.find(class_ = 'article-timestamp article-timestamp-updated')
  if updated is not None:
    updated = updated.find('time').attrs['datetime']
    updated = parser.parse(updated)
  else:
    updated = published

  # Get body
  # Ignore subheadings - inconsistent and not always super related to article e.g., it might check out this podcast
  # Ignore captions for photos (will just be repeats from article)
  paragraphs = soup.find_all('p', class_ = 'mol-para-with-font')

  body = ' '.join([p.get_text().strip() for p in paragraphs])

  # put everything in lower case for purposes of checking for dupes

  return {
      "headline": headline.lower(),
      "authors": authors,
      "published": published.timestamp(),
      "updated": updated.timestamp(),
      "body": body.lower(),
      "url": url
  }

def read_guardian_article(url):
  # Requests need the url of the page to access
  site = requests.get(url)

  # Can't validate URL for guardian because it doesn't contain special string for articles

  # Beautiful Soup takes the text from the page accessed by requests and makes it easy to use
  # 'html.parser' is the standard way to process HTML in python
  soup = BeautifulSoup(site.text, 'html.parser')

  err_obj = {
      'url': url
  }

  # Get headline - CHECK THAT THERE IS ONLY ONE HEADLINE
  headline = soup.find_all('h1')
  if len(headline) != 1:
    err_obj['failed'] = 'headline'
    return err_obj
  headline = headline[0].get_text().lower()


  # Validate authors
  authors = soup.find_all('a', {'rel': 'author'})

  if len(authors) != 0: authors = [author.get_text().strip().lower() for author in authors]

  # Sometimes authors stored elsewhere
  if len(authors) == 0:
    author = soup.find('meta', {'property': 'article:author'})
    if author is not None:
      authors = [author.attrs['content'].strip().lower()]


  if len(authors) == 0:
    err_obj['failed'] = 'author'
    return err_obj



  # Dailymail has published and updated date let's record both
  # Get published time
  published = soup.find('meta', {'property': 'article:published_time'})

  if published is None:
    err_obj['failed'] = 'published'
    return err_obj
  published = published.attrs['content']
  published = parser.parse(published)

  # Get updated time
  updated = soup.find('meta', {'property': 'article:modified_time'})

  if updated is not None:
    updated = updated.attrs['content']
    updated = parser.parse(updated)
  else:
    updated = published

  # Get body
  # Include subheadings - check if it exists first though

  subheading = soup.find_all(class_ = "dcr-1m3qdf6")
  if len(subheading) > 0:
    subheading = subheading[0].get_text().strip() + " "
  else:
    subheading = ""

  # Ignore captions for photos (will just be repeats from article)
  paragraphs = (soup.find_all('p', class_ = "dcr-s3ycb2"))

  body = subheading + ' '.join([p.get_text().strip() for p in paragraphs])

  # put everything in lower case for purposes of checking for dupes

  return {
      "headline": headline.lower(),
      "authors": authors,
      "published": published.timestamp(),
      "updated": updated.timestamp(),
      "body": body.lower(),
      "url": url
  }

#x = read_dailymail_article("https://www.dailymail.co.uk/news/article-14264907/Uber-Eats-different-prices-Coles-chicken.html")
#x = read_guardian_article("https://www.theguardian.com/technology/2017/jun/28/uber-drivers-deserve-to-be-treated-far-better")
print(x)

{'headline': 'uber drivers deserve to be treated far better', 'authors': ['letters'], 'published': 1498673931.0, 'updated': 1511816615.0, 'body': 'the charge sheet against travis kalanick, and the uber board as a whole, is a lengthy one (end of the road – uber’s investors decide chief is liability after avalanche of claims, 22 june). there is, however, one major omission. the rampant exploitation of uber’s tens of thousands of drivers under kalanick’s watch should be at the very top of that charge sheet. the fact that it hardly features is hugely concerning. an employment tribunal found last year that uber is wrongly depriving its drivers of basic protections such as the minimum wage, sick pay, and holiday pay – my own investigation found that some drivers take home as little as £4 an hour. likewise, transport for london has yet to grant uber a full renewal of its licence, due to concerns around its operations. now is the time to sweep away two of uber’s most egregious aspects: the bul

In [ ]:
# Using the read daily mail and read guardian functions - let's put our current list into a nicer format

def convert_all_results(paper):
  out = []
  loc = ""

  if paper == 'dailymail':
    loc = './raw_dailymail.json'
  elif paper == 'guardian':
    loc = './raw_guardian.json'
  else:
    return []

  with open(loc, 'r') as f:
    data = json.load(f)
    i = 0
    failed = 0
    for page in data:
      for article in page['organic_results']:
        i += 1
        if paper == 'dailymail': obj = read_dailymail_article(article['link'])
        elif paper == 'guardian': obj = read_guardian_article(article['link'])
        if 'failed' in obj:
          print(obj['failed'] + " : " + article['link'])
          failed += 1
        print(str(i) + " completed, " + str(failed) + " failed")


        out.append(obj)
  return out


articles = convert_all_results('guardian')

with open('clean_guardian.json', 'w') as f:
  json.dump(articles, f)

In [4]:
# This function merges the clean daily mail and guardian lists
def merge_clean_data(dailymail_loc, guardian_loc):
  merged = {
      'time': datetime.now().timestamp()
  }
  # Read dailymail and put into the merged object
  with open(dailymail_loc, 'r') as f:
    merged['dailymail'] = json.load(f)
  # Read guardian and put into the merged object
  with open(guardian_loc, 'r') as f:
    merged['guardian'] = json.load(f)

  return merged

# Store merged into a file
with open('./merged.json', 'w') as f:
  json.dump(merge_clean_data('./clean_dailymail', './clean_guardian'), f)

FileNotFoundError: [Errno 2] No such file or directory: './clean_dailymail'